In [ ]:
import warnings
warnings.filterwarnings('ignore',category=FutureWarning)
import warnings
warnings.filterwarnings("ignore")

import sys
#import os
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import h5py as h5
import sklearn
from sklearn.multioutput import MultiOutputClassifier
from sklearn import metrics
from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, average_precision_score, precision_recall_curve, accuracy_score, confusion_matrix  
from sklearn.metrics import average_precision_score
import pickle
import keras
from keras.models import load_model
import sys
sys.path.append(r'C:\Users\aoara\develop\deepbeat')
import utils
from pathlib import Path
from tensorflow.keras.optimizers import Adam
from scipy.io import loadmat
import copy
#import seaborn as sns


# Load model

In [ ]:
def get_training_config():
    with h5.File(r"C:\Users\aoara\develop\deepbeat\deepbeat.h5", 'r') as f:
    # Check for version attributes
        if 'keras_version' in f.attrs:
            print(f"Keras version: {f.attrs['keras_version']}")
        
        if 'backend' in f.attrs:
            print(f"Backend: {f.attrs['backend']}")
        
        # Sometimes stored under model config
        if 'model_config' in f.attrs:
            import json
            config = json.loads(f.attrs['model_config'])
            if 'keras_version' in config:
                print(f"Keras version (from config): {config['keras_version']}")
    
    training_config = json.loads(f.attrs['training_config'])
    return training_config 

In [ ]:
# get training config
with h5.File(r"C:\Users\aoara\develop\deepbeat\deepbeat.h5", 'r') as f:
    # Check for version attributes
    if 'keras_version' in f.attrs:
        print(f"Keras version: {f.attrs['keras_version']}")
    
    if 'backend' in f.attrs:
        print(f"Backend: {f.attrs['backend']}")
    
    # Sometimes stored under model config
    if 'model_config' in f.attrs:
        import json
        config = json.loads(f.attrs['model_config'])
        if 'keras_version' in config:
            print(f"Keras version (from config): {config['keras_version']}")
    
    # Print all available attributes
    print("\nAll attributes in file:")
    for key in f.attrs.keys():
        print(f"  {key}: {f.attrs[key]}")
    
    training_config = json.loads(f.attrs['training_config'])

In [ ]:
orig_config

In [ ]:
orig_config = training_config ['optimizer_config']['config']
orig_config['learning_rate'] = orig_config.pop('lr') # rename lr to learning rate
orig_config.pop('decay') # there is no longer a parameter called decay; the original decay was 0

# load deepbeat model with new tensorflow package, verify performances
path_to_model =r'C:\Users\aoara\develop\deepbeat'
model_name = 'deepbeat.h5'
deepbeat = load_model( Path(path_to_model) / model_name, compile = False) 

## Verify Loaded Model Performances

In [ ]:
## load original test data
data_path = Path(r'C:\Users\aoara\develop\deepbeat\data\db')
data_test = np.load(data_path / 'test.npz', allow_pickle=True)
test_x = data_test['signal']
test_qa = data_test['qa_label']
test_r = data_test['rhythm']
test_p = pd.DataFrame(data_test['parameters'])
print('test data shape: ')
print(test_x.shape)
print(test_qa.shape)
print(test_r.shape)
print(test_p.shape)
test_p.rename(index=str, columns={0:'timestamp', 
                                  1:'stream', 
                                  2:'ID'}, inplace=True)
## QA results

predictions_qa, predictions_r = deepbeat.predict(test_x)
predictions_QA = np.argmax(predictions_qa, axis=1)

#print(classification_report(np.argmax(test_qa, axis=1), predictions_QA))


excellent_qa_indx = np.where(predictions_QA==2)[0]
x_test_excellent = test_x[excellent_qa_indx,:]
p_test_excellent = test_p.iloc[excellent_qa_indx,:]
rhythm_test_excellent = test_r[excellent_qa_indx,:]
quality_assessment_test_excellent = test_qa[excellent_qa_indx,:]


# weighted macro-average across all indivduals

test_metrics_1 = utils.collecting_individual_metrics(deepbeat, x_test_excellent, p_test_excellent, rhythm_test_excellent, out_message=False)
test_metrics = pd.DataFrame.from_dict(test_metrics_1).T.rename(columns={0:'TPR', 1:'TNR', 2:'FPR', 3:'FNR', 4:"total_samples"})


for m in ['TPR', 'TNR', 'FPR', 'FNR']:
    metric_wmu = np.average(test_metrics[m][~test_metrics[m].isna()], weights=test_metrics['total_samples'][~test_metrics[m].isna()])
    print('%s: %0.2f' % (m, metric_wmu))
    
# PPV, NPV and F1

episode_m = utils.episode_metrics(deepbeat, x_test_excellent, p_test_excellent, rhythm_test_excellent, out_message=False)
episode_metrics = pd.DataFrame(episode_m).T
episode_metrics.rename(columns={0:'TPR', 1:'TNR', 2:'PPV', 3:'NPV', 4:"FPR", 5:'FNR', 6:'F1', 7:'total_samples'}, inplace=True)
for m in ['PPV', 'NPV', 'F1']:
    print('%s: %0.2f' % (m, episode_metrics[m]))

- benchmarks paper in PPG realms; see how it is constructed
-- see how they did it
-- publish this

- VSM watch version, algorithm versions

## Overview
### Step 1: Prepare data
1. deepbeat original + cleaned deepbeat data ---> took too long to update
- 1.1 for each updated subject, replace every updated labels --> took >3hr to find data matches and replace labels
- 1.2 for each updated subject, remove their old data, only use their new data
2. deepbeat original + cleaned deepbeat data + VSM
3. only with samiya's data

In [ ]:
def load_original_data(data_path, file_name):
    data = np.load(Path(data_path) / file_name,allow_pickle=True )
    output = {}
    output['data'] = data['signal']
    output['qa_label'] = data['qa_label']
    output['rhythm'] = data['rhythm'] 
    params = pd.DataFrame(data['parameters'])
    params.rename(index=str, columns={0:'timestamp', 
                                  1:'stream', 
                                  2:'ID'}, inplace=True)                            
    output['ID'] = np.array(params['ID'].to_list())
    return output

def load_from_mat(dir_path, file_name):
    file_mat = loadmat(Path(dir_path) / file_name)
    file = file_mat.get(file_name[:-4])
    return file 

def load_relabeled_data(data_path):
    # return combinbed, relabeled_db, relabeled_VSM
    ['data'], ['qa_label'], ['rhythm'], ['parameters'], ['ID']
    combined = {}
    combined['data'] = load_from_mat(data_path,'db_vsm_combined_data.mat' )
    combined['qa_label'] = load_from_mat(data_path, 'db_vsm_combined_label_q.mat' )
    combined['rhythm'] = load_from_mat(data_path, 'db_vsm_combined_label_r.mat' )
    combined['ID'] =load_from_mat(data_path, 'db_vsm_combined_sub_id.mat').flatten()
    # reshaping to original data
    # reshaping to match db's original data
    combined['data'] = combined['data'].reshape(combined['data'].shape[0], combined['data'].shape[1], 1)
    num_classes_rhythm = 2
    num_classes_qa = 3
    # one-hot encoding
    combined['rhythm']= keras.utils.to_categorical(combined['rhythm'], num_classes_rhythm)
    combined['qa_label'] = keras.utils.to_categorical(combined['qa_label'], num_classes_qa)
    
    relabeled_db = {}
    relabeled_vsm = {}
    
    # VSM index starts from 1000
    db_mask = (combined['ID'] < 1000).flatten()
    vsm_mask = (combined['ID']> 1000).flatten()
    # seperate the db data
    relabeled_db['data'] = combined['data'][db_mask,:]
    relabeled_db['qa_label'] = combined['qa_label'][db_mask, :]
    relabeled_db['rhythm'] = combined['rhythm'][db_mask, :]
    relabeled_db['ID'] =  combined['ID'][db_mask].flatten()
    # seperate the vsm data
    relabeled_vsm['data'] = combined['data'][vsm_mask, :]
    relabeled_vsm['qa_label'] = combined['qa_label'][vsm_mask, :]
    relabeled_vsm['rhythm'] = combined['rhythm'][vsm_mask,:]
    relabeled_vsm['ID'] = combined['ID'][vsm_mask].flatten()
    
    return combined, relabeled_db , relabeled_vsm
    

orig_data_path = Path(r'C:\Users\aoara\develop\deepbeat\data\db')
db_test = load_original_data(orig_data_path, 'test.npz')
db_train = load_original_data(orig_data_path, 'train.npz')

relabled_path = Path(r'C:\Users\aoara\develop\deepbeat\data\labeled_data')
relabeled_combined, relabeled_db, relabeled_vsm = load_relabeled_data(relabled_path)

# found that for each subject, number of relabeled data != number of original data
# so only a segment of original data got relabeled, need to find the ones in original data
# selected for relabeling, and then assign the relabeled rhythmns and quality labels back

#### why 1.1 is taking too long: 

In [ ]:
def compare_stats(relabeled_db,db_train):
    count_stats = []
    for num in np.unique(relabeled_db['ID'].flatten()):
        count_stats.append(
            [num, np.sum(db_train['ID'].flatten() ==num), 
        np.sum(relabeled_db['ID'].flatten() == num)]      
        )
    stats = pd.DataFrame(count_stats)
    stats.rename(index=str, columns={0:'subject id', 
                                    1:'original data count', 
                                    2:'relabeled data count'}, inplace=True) 
    return stats 

stats = compare_stats(relabeled_db,db_train)
stats.head(5)
    

In [ ]:
from keras.optimizers import Adam

# Try to create Adam with weight_decay
try:
    optimizer = Adam(learning_rate=0.001, weight_decay=1e-4)
    print("✅ weight_decay parameter is supported in Adam!")
except TypeError as e:
    print(f"❌ weight_decay NOT supported: {e}")


In [ ]:
# for each subject ID in relabeled_db['ID'], 
# find corresponding data, rhythmn label, quality label from original data
for sub_id in np.unique(relabeled_db['ID']):
    mask_relabeled = (relabeled_db['ID'] == sub_id)
    mask_origin = (db_train['ID'] == sub_id)
    
    # get original, relabeled data for each subject
    orig_data = db_train['data'][mask_origin, : ]
    orig_rhy = db_train['rhythm'][mask_origin, : ].copy()
    orig_qa = db_train['qa_label'][mask_origin, : ].copy()
    relabled_data = relabeled_db['data'][mask_relabeled, :]
    relabled_rhy = relabeled_db['rhythm'][mask_relabeled,: ]
    relabled_qa = relabeled_db['qa_label'][mask_relabeled,: ]
    
    # for each relabled data segment, find the corresponding one in original data,
    # use the mask to replace the original qa and rhythm label with relabled ones
    for seg_i in range(relabled_data.shape[0]):
        result_i = np.sum(orig_data - relabled_data[seg_i, :], axis = 1) 
        mask_replace = (result_i.flatten() == 0)
        
        if np.sum(mask_replace) == 1:
            orig_rhy [mask_replace, :] = relabled_rhy[seg_i, :]
            orig_qa [mask_replace, :] = relabled_qa[seg_i, :]
        elif np.sum(mask_replace) == 0:
            print(f"Warning: No match found for subject {sub_id}, segment {seg_i}")
        else:
            print(f"Warning: Multiple matches found for subject {sub_id}, segment {seg_i}")
            
    # assign this back to train_copy
    train_copy['rhythm'][mask_origin, : ] = orig_rhy
    db_train['qa_label'][mask_origin, : ] = orig_qa
        

In [ ]:
import sys
sys.path.append( r'C:\Users\aoara\develop\deepbeat')
from train_new_model import *
orig_data_path = r'C:\Users\aoara\datasets\db'
relabled_path = r'C:\Users\aoara\develop\deepbeat\data\labeled_data'
train_data = load_original_data(orig_data_path, 'train.npz')
relabeled_combined, relabeled_db, relabeled_vsm = load_relabeled_data(relabled_path)
# merged old data with relabeled data
#db_train_copy = copy.deepcopy(orig_train)
# db_train_update = replace_updated_subjects_db(db_train_copy, relabeled_db)
# db_VSM_train = attach_VSM(db_train_update, relabeled_vsm)

In [ ]:
set(np.unique(relabeled_db['ID'])) &set(np.unique(db_train['ID'])) & set(np.unique(db_val['ID']))

In [ ]:
def find_and_replace_relabled_signals(db_train, relabeled_db):
    # Group operations by subject ID to avoid repeated masking
    for sub_id in np.unique(relabeled_db['ID']):
        mask_relabeled = (relabeled_db['ID'] == sub_id)
        mask_origin = (db_train['ID'] == sub_id)
        
        # Get original and relabeled data for each subject
        orig_data = db_train['data'][mask_origin, :]
        orig_rhy = db_train['rhythm'][mask_origin, :].copy()
        orig_qa = db_train['qa_label'][mask_origin, :].copy()
        relabeled_data = relabeled_db['data'][mask_relabeled, :]
        relabeled_rhy = relabeled_db['rhythm'][mask_relabeled, :]
        relabeled_qa = relabeled_db['qa_label'][mask_relabeled, :]
        
        # VECTORIZED: Compare all relabeled segments against all original segments at once
        # Shape: (n_relabeled, n_original, n_features) -> (n_relabeled, n_original)
        differences = np.sum(orig_data[np.newaxis, :, :] - relabeled_data[:, np.newaxis, :], axis=2)
        
        # Find matches: where difference sum equals 0
        # Shape: (n_relabeled, n_original) - boolean array
        matches = (differences == 0)
        
        # Count matches per relabeled segment
        match_counts = np.sum(matches, axis=1)
        
        # Process based on match counts
        single_match = (match_counts == 1)
        no_match = (match_counts == 0)
        multiple_match = (match_counts > 1)
        
        # Warnings for problematic cases
        if np.any(no_match):
            no_match_indices = np.where(no_match)[0]
            for seg_i in no_match_indices:
                print(f"Warning: No match found for subject {sub_id}, segment {seg_i}")
        
        if np.any(multiple_match):
            multiple_match_indices = np.where(multiple_match)[0]
            for seg_i in multiple_match_indices:
                print(f"Warning: Multiple matches found for subject {sub_id}, segment {seg_i}")
        
        # Update labels where we have exactly one match
        # For each relabeled segment with single match, find which original segment it matched
        for seg_i in np.where(single_match)[0]:
            orig_idx = np.where(matches[seg_i])[0][0]  # Get the index of the match
            orig_rhy[orig_idx, :] = relabeled_rhy[seg_i, :]
            orig_qa[orig_idx, :] = relabeled_qa[seg_i, :]
        
        # Assign back to db_train
        db_train['rhythm'][mask_origin, :] = orig_rhy
        db_train['qa_label'][mask_origin, :] = orig_qa

    return db_train


In [ ]:
from tqdm import tqdm

def chunk_and_replace_relabeled_signals(db_train, relabeled_db, chunk_size=100):
    """
    Memory-efficient version that processes relabeled segments in chunks
    """
    unique_subjects = np.unique(relabeled_db['ID'])
    
    for sub_id in tqdm(unique_subjects, desc="Processing subjects"):
        
        mask_relabeled = (relabeled_db['ID'] == sub_id)
        mask_origin = (db_train['ID'] == sub_id)
        
        # Get original and relabeled data for each subject
        orig_data = db_train['data'][mask_origin, :]
        orig_rhy = db_train['rhythm'][mask_origin, :].copy()
        orig_qa = db_train['qa_label'][mask_origin, :].copy()
        relabeled_data = relabeled_db['data'][mask_relabeled, :]
        relabeled_rhy = relabeled_db['rhythm'][mask_relabeled, :]
        relabeled_qa = relabeled_db['qa_label'][mask_relabeled, :]
        
        n_relabeled = relabeled_data.shape[0]
        n_original = orig_data.shape[0]
        
        # Process in chunks to avoid memory issues
        for chunk_start in range(0, n_relabeled, chunk_size):
            chunk_end = min(chunk_start + chunk_size, n_relabeled) 
            
            # Process only a chunk of relabeled segments at a time
            chunk_data = relabeled_data[chunk_start:chunk_end, :]
            
            # Compare this chunk against all original segments
            # Shape: (chunk_size, n_original, n_features) -> (chunk_size, n_original)
            differences = np.sum(
                orig_data[np.newaxis, :, :] - chunk_data[:, np.newaxis, :], 
                axis=2
            )
            
            # Find matches
            matches = (differences == 0)
            match_counts = np.sum(matches, axis=1)
            
            # Process based on match counts
            single_match = (match_counts == 1)
            no_match = (match_counts == 0)
            multiple_match = (match_counts > 1)
            
            # Warnings
            if np.any(no_match):
                no_match_indices = np.where(no_match)[0] + chunk_start
                for seg_i in no_match_indices:
                    tqdm.write(f"Warning: No match found for subject {sub_id}, segment {seg_i}")
            
            if np.any(multiple_match):
                multiple_match_indices = np.where(multiple_match)[0] + chunk_start
                for seg_i in multiple_match_indices:
                    tqdm.write(f"Warning: Multiple matches found for subject {sub_id}, segment {seg_i}")
            
            # Update labels where we have exactly one match
            for local_idx in np.where(single_match)[0]:
                seg_i = chunk_start + local_idx
                orig_idx = np.where(matches[local_idx])[0][0]
                orig_rhy[orig_idx, :] = relabeled_rhy[seg_i, :]
                orig_qa[orig_idx, :] = relabeled_qa[seg_i, :]
        
        # Assign back to db_train
        db_train['rhythm'][mask_origin, :] = orig_rhy
        db_train['qa_label'][mask_origin, :] = orig_qa
    
    return db_train

In [ ]:
db_train_replaced = chunk_and_replace_relabeled_signals(train_data , relabeled_db, chunk_size= 30)

In [ ]:
np.unique(relabeled_db['ID'])

In [ ]:
output_dir = r'C:\Users\aoara\develop\deepbeat\output\replace_relabeled.pkl'
import pickle as pk
with open(output_dir, 'wb') as file:
    pickle.dump(db_train_replaced, file)

In [ ]:
check_ID  = [90,92,  93,  95,  99, 100, 102, 103, 104, 109, 111, 114, 115, 116,117, 122, 129, 130, 132, 134, 137]

In [ ]:
db_train_replaced.keys()

In [ ]:
python train_new_model.py  --file_name db_ --training_choice db_relabel  --output_path "C:\Users\aoara\OneDrive\Documents\repos\deepbeat\training_output" 

##### 1.2 for updated subjects, keep their relabeled data only

In [ ]:
def replace_updated_subjects_db(db_train, relabeled_db):
    
    subjects_to_replace = np.unique(relabeled_db['ID'])
    mask_keep = ~np.isin(db_train['ID'], subjects_to_replace)
    
    db_train['data'] = db_train['data'][mask_keep]
    db_train['rhythm'] = db_train['rhythm'][mask_keep]
    db_train['qa_label'] = db_train['qa_label'][mask_keep]
    db_train['ID'] = db_train['ID'][mask_keep]
    
    db_train['data'] = np.concatenate([db_train['data'], relabeled_db['data']], axis=0)
    db_train['rhythm'] = np.concatenate([db_train['rhythm'], relabeled_db['rhythm']], axis=0)
    db_train['qa_label'] = np.concatenate([db_train['qa_label'], relabeled_db['qa_label']], axis=0)
    db_train['ID'] = np.concatenate([db_train['ID'], relabeled_db['ID']], axis=0)
     
    return db_train

def attach_VSM (db_data, relabeled_vsm):
    db_data['data'] = np.concatenate([db_data['data'], relabeled_vsm['data']], axis=0)
    db_data['rhythm'] = np.concatenate([db_data['rhythm'], relabeled_vsm['rhythm']], axis=0)
    db_data['qa_label'] = np.concatenate([db_data['qa_label'], relabeled_vsm['qa_label']], axis=0)
    db_data['ID'] = np.concatenate([db_data['ID'], relabeled_vsm['ID']], axis=0)
    return db_data
    


In [ ]:
def check_array_health(arr, name):
    """Check an array for common issues"""
    print(f"\n{'='*60}")
    print(f"Checking: {name}")
    print(f"{'='*60}")
    
    print(f"Shape: {arr.shape}")
    print(f"Dtype: {arr.dtype}")
    print(f"Min: {np.min(arr)}")
    print(f"Max: {np.max(arr)}")
    print(f"Mean: {np.mean(arr)}")
    print(f"Std: {np.std(arr)}")
    
    # Check for NaN
    nan_count = np.sum(np.isnan(arr))
    print(f"NaN count: {nan_count} ({100*nan_count/arr.size:.4f}%)")
    
    # Check for Inf
    inf_count = np.sum(np.isinf(arr))
    print(f"Inf count: {inf_count} ({100*inf_count/arr.size:.4f}%)")
    
    # Check for zeros
    zero_count = np.sum(arr == 0)
    print(f"Zero count: {zero_count} ({100*zero_count/arr.size:.4f}%)")
    
    # Check for very large values
    very_large = np.sum(np.abs(arr) > 1e10)
    print(f"Very large values (>1e10): {very_large}")
    
    # Check for very small values (but not zero)
    very_small = np.sum((np.abs(arr) < 1e-10) & (arr != 0))
    print(f"Very small values (<1e-10, non-zero): {very_small}")
    
    return {
        'has_nan': nan_count > 0,
        'has_inf': inf_count > 0,
        'has_very_large': very_large > 0,
        'has_very_small': very_small > 0
    }

In [ ]:

#db_train_update = replace_updated_subjects_db(copy.deepcopy(db_train), relabeled_db)

db_VSM_train = attach_VSM(db_train_update, relabeled_vsm)

### Step 2: Construct Original deepbeat model in Pytorch
Did not have the data Jessica used for pretraining....
Will follow same training paramters Samiya used for the compact model

### Step 3: retrain deepbeat and report results

In [ ]:
def shuffle_data(db_train):
    """

    Args:
        db_train (dict): keys - 'data', 'qa_label', 'rhythm', 'ID'
    """
    data_train, label_train_r, label_train_q = db_train['data'], db_train['rhythm'], db_train['qa_label']
    # random shuffle
    idx = np.random.permutation(range(len(label_train_r)))  # shuffled indices
    # shuffle together
    data_train, label_train_r, label_train_q = data_train[idx, :], label_train_r[idx], label_train_q[idx]
    
    return data_train, label_train_r, label_train_q

In [ ]:
new_db = keras.models.clone_model(deepbeat)
new_db.compile(
    optimizer= Adam(
        **orig_config
    ),
    loss={
        'qa_output': 'categorical_crossentropy',
        'rhythm_output': 'binary_crossentropy' ### Samiya used BinaryFocalLoss(gamma=2)
    },
    loss_weights={
        'qa_output': 0.2,      
        'rhythm_output': 5.0   
    },
    metrics={'rhythm_output': 'accuracy', 'qa_output': 'accuracy'}
)

#  How Samiya compiled the compact DB :
# model.compile(optimizer=tf.keras.optimizers.Adam(),
#             loss={'rhythm_output': BinaryFocalLoss(gamma=2), 'qa_output': 'categorical_crossentropy'},
#             loss_weights={'rhythm_output': 1, 'qa_output': 1},
#             metrics={'rhythm_output': 'accuracy', 'qa_output': 'accuracy'})


In [ ]:
.\conda\envs\db_env\python train_new_model.py

In [ ]:
new_db.save(output_path / 'test_model.keras')

In [ ]:
output_path = Path(r'C:\Users\aoara\OneDrive\Documents\repos\deepbeat\new_models')

In [ ]:
data_train, label_train_r, label_train_q = shuffle_data(db_train)
batch_size = 128
epochs = 100
# Train Model
history = new_db.fit(data_train, {"rhythm_output": label_train_r, "qa_output": label_train_q},
          batch_size=batch_size,
          epochs=epochs,
          verbose=1)

output_path = Path(r'C:\Users\aoara\OneDrive\Documents\repos\deepbeat\new_folder')
# save model
new_db.save('.keras')
# save history


In [ ]:
import pickle

In [ ]:
predicted_values_r, predicted_values_q = new_db.predict(data_test)
label_pred_r = np.argmax(predicted_values_r, axis=1)
label_pred_q = np.argmax(predicted_values_q, axis=1)

In [ ]:
# Confusion Matrix (signal quality)
confusion_mtx = confusion_matrix(label_test_q, label_pred_q, normalize='true')
confusion_mtx = confusion_mtx * 100
# confusion_mtx = confusion_mtx.astype(int)
tick_pics = ['0', '1', '2']
plt.figure(figsize=(12, 10))
ax = sns.heatmap(confusion_mtx, annot=True, fmt="0.1f", cbar=True,
                    xticklabels=['poor', 'acceptable', 'excellent'],
                    yticklabels=['poor', 'acceptable', 'excellent'],
                    annot_kws={"size": 20})
plt.xlabel('Predicted value', fontsize=30, labelpad=10)
plt.ylabel('True value', fontsize=30, labelpad=10)
plt.title('Confusion Matrix (%)', fontsize=30)
ax.tick_params(labelsize=20)
plt.show()


# Confusion Matrix (signal rhythm)
# (change the label_test and label_pred for excellent/acceptable samples, e.g., label_test_r_exc, label_pred_r_exc)
confusion_mtx = confusion_matrix(label_test_r, label_pred_r, normalize='true')
confusion_mtx = confusion_mtx * 100
# confusion_mtx = confusion_mtx.astype(int)
tick_pics = ['0', '1']
plt.figure(figsize=(12, 10))
ax = sns.heatmap(confusion_mtx, annot=True, fmt="0.1f", cbar=True,
                    xticklabels=['sinus', 'afib'],
                    yticklabels=['sinus', 'afib'],
                    annot_kws={"size": 20})
plt.xlabel('Predicted value', fontsize=30, labelpad=10)
plt.ylabel('True value', fontsize=30, labelpad=10)
plt.title('Confusion Matrix (%)', fontsize=30)
ax.tick_params(labelsize=20)
plt.show()

In [ ]:
# cleaning datasets
